<a href="https://colab.research.google.com/github/Sg134-ch/Flyrank-ML-1/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")


In [3]:
rel = "hf://datasets/FlyRank/internship-warehouse"

con.sql(f"""
SELECT COUNT(*)
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘

In [4]:
rel = "hf://datasets/FlyRank/internship-warehouse"

con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
LIMIT 1
""")

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

In [5]:
con.sql(f"""
SELECT DISTINCT month
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
ORDER BY month
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┐
│  month  │
│ varchar │
├─────────┤
│ 2025-01 │
│ 2025-02 │
│ 2025-03 │
│ 2025-04 │
│ 2025-05 │
│ 2025-06 │
│ 2025-07 │
│ 2025-08 │
│ 2025-09 │
│ 2025-10 │
│ 2025-11 │
│ 2025-12 │
│ 2026-01 │
│ 2026-02 │
│ 2026-03 │
│ 2026-04 │
│ 2026-05 │
│ 2026-06 │
├─────────┤
│ 18 rows │
└─────────┘

# 1. Unit of Analysis + Time Window

### Unit of Analysis

One row represents the daily search and website performance metrics for a single webpage (`content_hash_id`) belonging to a specific client (`client_hash_id`) on a particular reporting date (`report_date`).

### Time Window

For this assignment I use **March 2026 (`month = '2026-03'`)**, which is a mid-panel month. This avoids using the final month (June 2026), which should remain a future evaluation period rather than a development period.

### Verification

The warehouse contains:

- **9,841,378 rows**
- **331,437 unique webpages**
- **55 unique clients**
- Reporting dates from **2026-03-01** to **2026-03-31**

These query results confirm that the selected slice represents daily webpage performance during March 2026.

# 2. Fields: Feature / Label / Context / Excluded

## Features

- gsc_impressions
- gsc_clicks
- gsc_sum_position
- sessions_organic
- scroll_events

These are observable signals available before making a refresh decision.

---

## Label / Proxy

A future decline indicator (or another historical performance proxy) will be used as the supervised learning target.

The warehouse itself does not directly contain a "needs refresh" label, so a proxy must be created from historical outcomes.

---

## Context Fields

- report_date
- client_hash_id
- content_hash_id
- month

These fields describe each observation but are not direct predictive features.

---

## Excluded Fields

Examples include:

- Future outcome variables
- Label-derived metrics
- Any field that would only become available after the decision point

These are excluded to prevent target leakage.

# 3. Verify it with Queries

### Query 1 — Grain and Time Window

Verified that March 2026 contains:

- 9,841,378 observations
- 331,437 webpages
- 55 clients
- Date range:
  2026-03-01 → 2026-03-31

---

### Query 2 — Data Availability

Availability check using `IS TRUE` shows:

- Total rows:
  9,841,378

- GSC available:
  3,611,061

- GA4 available:
  413,966

This demonstrates that not every row contains complete data and availability must be considered during feature engineering.

---

### Query 3 — Feature Frame

The feature frame contains:

- gsc_impressions
- gsc_clicks
- gsc_sum_position
- sessions_organic
- scroll_events

These variables are observable before the prediction is made and therefore suitable as candidate model features.

In [6]:
rel = "hf://datasets/FlyRank/internship-warehouse"

con.sql(f"""
SELECT
    COUNT(*) AS rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date,
    COUNT(DISTINCT content_hash_id) AS unique_pages,
    COUNT(DISTINCT client_hash_id) AS unique_clients
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03'
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬────────────┬────────────┬──────────────┬────────────────┐
│  rows   │ start_date │  end_date  │ unique_pages │ unique_clients │
│  int64  │    date    │    date    │    int64     │     int64      │
├─────────┼────────────┼────────────┼──────────────┼────────────────┤
│ 9841378 │ 2026-03-01 │ 2026-03-31 │       331437 │             55 │
└─────────┴────────────┴────────────┴──────────────┴────────────────┘

In [7]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03'
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬───────────────┬───────────────┐
│ total_rows │ gsc_available │ ga4_available │
│   int64    │    int128     │    int128     │
├────────────┼───────────────┼───────────────┤
│    9841378 │       3611061 │        413966 │
└────────────┴───────────────┴───────────────┘

In [9]:
con.sql(f"""
SELECT
    report_date,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_sum_position,
    sessions_organic,
    scroll_events
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month='2026-03'
LIMIT 10
""")

┌─────────────┬──────────────────────────┬─────────────────┬────────────┬──────────────────┬──────────────────┬───────────────┐
│ report_date │     content_hash_id      │ gsc_impressions │ gsc_clicks │ gsc_sum_position │ sessions_organic │ scroll_events │
│    date     │         varchar          │      int64      │   int64    │      int64       │      int64       │     int64     │
├─────────────┼──────────────────────────┼─────────────────┼────────────┼──────────────────┼──────────────────┼───────────────┤
│ 2026-03-01  │ content_b7e512995f79d5a6 │              20 │          0 │               67 │             NULL │          NULL │
│ 2026-03-01  │ content_05597932fe4da067 │               1 │          0 │                0 │             NULL │          NULL │
│ 2026-03-01  │ content_7a105f548d9c6916 │             125 │          1 │              616 │             NULL │          NULL │
│ 2026-03-01  │ content_905aa32a0230694e │               7 │          0 │               28 │            

In [10]:
con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""")

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

# 4. Data Limits

Although the warehouse provides extensive daily search performance information, it also has several limitations.

- Not every row has Google Search Console or GA4 data available.
- The warehouse cannot explain why search performance changes; it only records observed behaviour.
- Historical data may contain seasonality and client-specific differences.
- Some important content quality factors (writing quality, user intent, topical relevance) are not directly represented.
- The selected month represents only one period of time and should not be assumed to generalize perfectly across all future periods.

Therefore, the resulting model should be viewed as a decision-support system rather than a system that predicts search engine behaviour.

# Self-Check

- ✅ Unit of analysis clearly defined.
- ✅ Time window verified using SQL.
- ✅ Features, labels, context fields, and excluded fields identified.
- ✅ Three verification queries executed.
- ✅ Availability checked using `IS TRUE`.
- ✅ Five observable features selected.
- ✅ Data limitations documented.
- ✅ Notebook runs successfully.
- ✅ Claims use careful, observational language.